# Exercise 10 — Multi-factor models

Lecture 10 made two claims. That a factor is *constructed* rather than discovered, and
that the choice of benchmark decides whether a return is alpha or style. This notebook
shows both, on the one anomaly the CAPM fails most visibly: the flat security market line.
It sorts on beta, builds a betting-against-beta factor out of the failure, and then asks
whether that factor is anything new.

**Data.** Ten value-weighted portfolios formed on beta, the Fama-French five factors and
the momentum factor, all monthly, July 1963 to June 2026. Source:
[Kenneth R. French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html).

**How to work with this notebook.** This is a worked example: every cell is complete.
Each task states a question, the code under it answers it, and the commentary after the
code says what the result means and where people usually go wrong. Before you run a code
cell, read the task and decide what you expect to see. The **Your turn** block at the end
asks three questions that need no new code: you change one input at a time and read what
moves.

Run **Runtime → Restart and run all** before you trust any number in here.

In [ ]:
!wget -q https://raw.githubusercontent.com/KroeTiA/Investments/main/exercise_utils.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter

from exercise_utils import FHNW, setup_style, save_results
setup_style()

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")


def as_percent(df, cols):
    # display copy with the named columns scaled to percentage points
    out = df.copy()
    out[cols] = out[cols] * 100
    return out.to_string(float_format=lambda v: f"{v:8.2f}")

### The data

The French files are committed verbatim, copyright lines and all, so the cleaning happens
here rather than somewhere upstream. Each file stacks several blocks on top of each
other, so we start at the header row of the block we want and stop at the first row that
is not a `YYYYMM` date. Returns arrive in percentage points and are converted to
decimals; missing values are coded `-99.99` or `-999`.

`START` is the one place the sample start is set. Everything downstream reads it.

In [ ]:
BASE = ("https://raw.githubusercontent.com/KroeTiA/Investments/main/"
        "Exercise_10/data/")


def read_french(url, skiprows):
    # read one block of a Kenneth French CSV into monthly decimal returns
    df = pd.read_csv(url, skiprows=skiprows)
    df = df.rename(columns={df.columns[0]: "date"})
    df["date"] = df["date"].astype(str).str.strip()
    is_month = df["date"].str.fullmatch(r"\d{6}")
    stop = (~is_month).values.argmax() if (~is_month).any() else len(df)
    df = df.iloc[:stop]
    df = df.set_index(pd.PeriodIndex(df["date"], freq="M")).drop(columns="date")
    df.columns = [c.strip() for c in df.columns]
    return df.astype(float).replace([-99.99, -999.0], np.nan) / 100


# skiprows points at the header line of the block we want
ports = read_french(BASE + "Portfolios_Formed_on_BETA.csv", 15)   # VW, monthly
ff5 = read_french(BASE + "F-F_Research_Data_5_Factors_2x3.csv", 4)
mom = read_french(BASE + "F-F_Momentum_Factor.csv", 13)

DEC = ["Lo 10", "Dec 2", "Dec 3", "Dec 4", "Dec 5",
       "Dec 6", "Dec 7", "Dec 8", "Dec 9", "Hi 10"]

START = "1963-07"      # first month of the sample

data = ports[DEC].join(ff5, how="inner").join(mom, how="inner").dropna()
data = data.loc[START:]
rx = data[DEC].sub(data["RF"], axis=0)      # excess returns of the ten deciles
mkt = data["Mkt-RF"]                        # the market factor

print(f"{len(data)} months, {data.index[0]} to {data.index[-1]}")
data.head(3)

## Task 1 — Does the CAPM survive a sort on beta?

Every June, French sorts the whole US market into ten value-weighted portfolios by
estimated beta. `rx` holds their monthly excess returns. If the CAPM held, the ten
average excess returns would sit on a straight line through the origin whose slope is
the market risk premium.

**(a)** Build one table, `capm`, with a row per decile and these columns: annualised
mean excess return, annualised volatility, annualised Sharpe ratio, the full-sample
beta, the annualised CAPM alpha, and the t-statistic of the monthly alpha.

**(b)** Plot the ten portfolios in beta–mean-return space. Draw the theoretical
security market line through the origin and the market portfolio, and fit a line
through the ten points. Report the fitted intercept and slope next to their
theoretical values.

Then say which alphas you would call economically large, and whether their signs match
what the lecture predicted.

In [ ]:
# for each decile: regress its excess return on the market excess return, and collect
# mean, volatility, Sharpe ratio, beta, alpha (annualised) and t(alpha)
rows = {}
for p in DEC:
    res = sm.OLS(rx[p], sm.add_constant(mkt)).fit()
    rows[p] = {
        "mean": rx[p].mean() * 12,
        "sd": rx[p].std() * np.sqrt(12),
        "sr": rx[p].mean() / rx[p].std() * np.sqrt(12),
        "beta": res.params.iloc[1],
        "alpha": res.params.iloc[0] * 12,
        "t(alpha)": res.tvalues.iloc[0],
    }

capm = pd.DataFrame(rows).T
print("mean, sd and alpha in % p.a.\n")
print(as_percent(capm, ["mean", "sd", "alpha"]))

rp_market = mkt.mean() * 12
print(f"\nmarket risk premium {rp_market:.2%} p.a., "
      f"volatility {mkt.std() * np.sqrt(12):.2%}, "
      f"Sharpe {mkt.mean() / mkt.std() * np.sqrt(12):.3f}")

In [ ]:
# the empirical SML: regress the ten mean returns on the ten betas
sml = sm.OLS(capm["mean"], sm.add_constant(capm["beta"])).fit()
print(f"fitted   intercept {sml.params.iloc[0]:+.2%} (t {sml.tvalues.iloc[0]:.2f}), "
      f"slope {sml.params.iloc[1]:+.2%} (t {sml.tvalues.iloc[1]:.2f}), "
      f"R2 {sml.rsquared:.3f}")
print(f"theory   intercept  0.00%,          slope {rp_market:+.2%}")

# the ten portfolios, the theoretical line and the fitted line
fig1, ax = plt.subplots(figsize=(7, 4.5))
grid = np.linspace(0, 1.75, 50)
ax.plot(grid, rp_market * grid, color=FHNW["navy"], lw=1.6, label="theoretical SML")
ax.plot(grid, sml.params.iloc[0] + sml.params.iloc[1] * grid,
        color=FHNW["red"], lw=1.6, ls="--", label="fitted line")
ax.scatter(capm["beta"], capm["mean"], s=45, color=FHNW["blue"], zorder=3,
           label="beta deciles")
ax.scatter([1.0], [rp_market], s=90, marker="*", color=FHNW["green"],
           zorder=4, label="market")
for p, off in [("Lo 10", (-4, 10)), ("Hi 10", (8, -4))]:
    ax.annotate(p, (capm.loc[p, "beta"], capm.loc[p, "mean"]),
                textcoords="offset points", xytext=off, fontsize=9)
ax.set_xlabel("beta")
ax.set_ylabel("mean excess return, p.a.")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.legend(frameon=False)
plt.show()

### What Task 1 shows

The Sharpe ratio falls from **0.55** in the lowest-beta decile to **0.31** in the highest
(market: **0.47**): more market risk was not rewarded. The alphas run from
**+2.37 % p.a. (t = 2.41)** at the bottom to **−2.71 % p.a. (t = −1.52)** at the top.

The fitted line has intercept **+4.24 %** where theory says zero, and slope **+3.66 %**
where theory says the market risk premium of **7.21 %**. Beta does sort returns
(R² = 0.67), but the line is about half as steep as it should be — Black, Jensen and
Scholes (1972), still there sixty years later.

The mistake to expect: reading the flat line as "beta is not priced". It is priced, just
not at the rate the CAPM demands — the gap a second factor is invented to fill.

## Task 2 — Build a factor out of the failure

The lecture's claim: a factor is not discovered in the data, it is constructed. Task 1
handed you a characteristic that sorts average returns the wrong way. Turn it into a
portfolio.

**(a)** The obvious attempt first. Go long one dollar of the lowest-beta decile and
short one dollar of the highest. Report its mean excess return, volatility, beta and
alpha.

**(b)** Now make it market-neutral. Lever the low-beta leg up and the high-beta leg
down so that each carries a beta of exactly one before you subtract: hold
$1/\beta_L$ in the long leg and $1/\beta_H$ in the short leg, with the difference
borrowed or lent at the risk-free rate. Call the result `bab`. Confirm its beta is zero
and report its Sharpe ratio and alpha.

**(c)** Plot the compounded excess return of `bab` against the market.

Then answer the question the comparison raises: the naive portfolio has a **negative**
average return and a **positive** alpha. How can both be true at once?

In [ ]:
LONG, SHORT = "Lo 10", "Hi 10"      # the two legs

# the naive long-short, then the two beta-neutralising weights, then bab
naive = rx[LONG] - rx[SHORT]

w_long = 1 / capm.loc[LONG, "beta"]
w_short = 1 / capm.loc[SHORT, "beta"]
bab = w_long * rx[LONG] - w_short * rx[SHORT]

print(f"long  {w_long:.3f} x {LONG}   (beta {capm.loc[LONG, 'beta']:.3f})")
print(f"short {w_short:.3f} x {SHORT}   (beta {capm.loc[SHORT, 'beta']:.3f})")
print(f"net exposure to the risk-free asset: {w_short - w_long:+.3f}\n")


def summarise(r, label):
    # mean, volatility, Sharpe ratio, beta, alpha and t(alpha) of one return series
    m = mkt.reindex(r.index)
    res = sm.OLS(r, sm.add_constant(m)).fit()
    return pd.Series({
        "mean": r.mean() * 12,
        "sd": r.std() * np.sqrt(12),
        "sr": r.mean() / r.std() * np.sqrt(12),
        "beta": res.params.iloc[1],
        "alpha": res.params.iloc[0] * 12,
        "t(alpha)": res.tvalues.iloc[0],
    }, name=label)


factors = pd.DataFrame([summarise(naive, "naive long - short"),
                        summarise(bab, "BAB")])
print("mean, sd and alpha in % p.a.\n")
print(as_percent(factors, ["mean", "sd", "alpha"]))

In [ ]:
# compound both excess return series and plot them on a log scale
wealth = pd.DataFrame({"BAB": (1 + bab).cumprod(),
                       "market": (1 + mkt).cumprod()})

fig2, ax = plt.subplots(figsize=(7, 4.5))
wealth.plot(ax=ax, color=[FHNW["blue"], FHNW["navy"]], lw=1.4)
ax.set_yscale("log")
ax.set_xlabel("")
ax.set_ylabel(f"compounded excess return, 1 = {data.index[0]}")
ax.legend(frameon=False)
plt.show()

drawdown = wealth["BAB"] / wealth["BAB"].cummax() - 1
print(f"BAB worst drawdown {drawdown.min():.1%}, trough {drawdown.idxmin()}")

### What Task 2 shows

The weights are **1.681** on the long leg and **0.624** on the short leg, so the strategy
borrows **1.057** at the risk-free rate: BAB is a levered position, which is Frazzini and
Pedersen's point — investors who cannot borrow bid up high-beta stocks instead.

| | mean p.a. | vol p.a. | beta | alpha p.a. | t |
|---|---|---|---|---|---|
| naive long − short | **−2.18 %** | 24.7 % | **−1.01** | **+5.08 %** | 2.08 |
| BAB | **+5.67 %** | 19.1 % | **0.00** | +5.67 % | 2.34 |

The naive portfolio *lost* money on average and still had a positive alpha: with a beta
of −1.01 the CAPM required it to lose about 7 % a year. **Alpha is a residual, not a
return.** Once the market exposure is gone, mean and alpha coincide.

Two caveats. The betas come from the full sample, so the neutrality holds only with
hindsight. And the figure shows why nobody holds this alone: BAB fell **53.6 %** into
February 2000, when high-beta technology stocks ran away from everything else.

## Task 3 — A two-factor model

A factor model is judged by how well it explains a set of test assets. And ideally, a
new factor is not already captured by the existing, well-known ones.

**(a)** Add `bab` to the market and re-run Task 1's ten regressions, now with two
regressors. Put the two-factor alphas and t-statistics next to the CAPM ones, and report
the mean absolute alpha under each model.

**(b)** Now put `bab` on the left-hand side. Regress it on the market alone, then on
FF3 (`Mkt-RF, SMB, HML`), then on FF5 (adds `RMW, CMA`), then on FF5 plus `Mom`. A
regression of one factor on a set of others is a **spanning test**: it asks whether the
returns of your factor could have been reproduced by holding a combination of theirs.
An alpha that survives means it could not.

Then: which model kills BAB's alpha, and which factors do most of the work? And in (a),
which two deciles were always going to lose their alpha, whatever the data had looked
like?

**New this week: regression with several factors.** `sm.OLS` takes a DataFrame of
regressors exactly as it took a single series. The intercept is still alpha, but it is
now measured against a benchmark built from all of them.

In [ ]:
# regress each decile on [market, bab] and compare the alphas to Task 1's
X2 = pd.concat([mkt, bab.rename("BAB")], axis=1)

rows = {}
for p in DEC:
    res = sm.OLS(rx[p], sm.add_constant(X2)).fit()
    rows[p] = {
        "alpha CAPM": capm.loc[p, "alpha"],
        "t CAPM": capm.loc[p, "t(alpha)"],
        "alpha 2F": res.params.iloc[0] * 12,
        "t 2F": res.tvalues.iloc[0],
        "beta BAB": res.params.iloc[2],
    }

two_factor = pd.DataFrame(rows).T
print("alphas in % p.a.\n")
print(as_percent(two_factor, ["alpha CAPM", "alpha 2F"]))
print(f"\nmean |alpha|   CAPM {two_factor['alpha CAPM'].abs().mean():.2%}"
      f"   Mkt+BAB {two_factor['alpha 2F'].abs().mean():.2%}")

In [ ]:
# spanning tests: regress bab on each of the four factor sets in turn
MODELS = {
    "CAPM": ["Mkt-RF"],
    "FF3": ["Mkt-RF", "SMB", "HML"],
    "FF5": ["Mkt-RF", "SMB", "HML", "RMW", "CMA"],
    "FF5 + MOM": ["Mkt-RF", "SMB", "HML", "RMW", "CMA", "Mom"],
}

rows = []
for name, cols in MODELS.items():
    res = sm.OLS(bab, sm.add_constant(data[cols])).fit()
    row = {"alpha": res.params.iloc[0] * 12,
           "t(alpha)": res.tvalues.iloc[0],
           "R2": res.rsquared}
    row.update({c: res.params[c] for c in cols})
    rows.append(pd.Series(row, name=name))

spanning = pd.DataFrame(rows)
print("alpha in % p.a., loadings in units of the factor\n")
print(as_percent(spanning, ["alpha"]))

### What Task 3 shows

**(a)** Mean absolute alpha falls from **1.09 % to 0.79 %**, and the two alphas that
mattered are gone: `Lo 10` from +2.37 % to +0.25 %, `Hi 10` from −2.71 % to +0.67 %. But
`bab` is built from exactly those two portfolios, so their alphas had to collapse. The
informative part is elsewhere: `Dec 8` and `Dec 9` *acquire* alphas of +1.67 % and
+2.38 % (t = 1.95 and 2.26). A factor built from the extremes of a sort always prices
those extremes; the test is what it does to everything else.

**(b)** The spanning test:

| model | alpha p.a. | t | R² |
|---|---|---|---|
| CAPM | +5.67 % | 2.34 | 0.00 |
| FF3 | +3.92 % | 1.90 | 0.29 |
| **FF5** | **+0.22 %** | **0.11** | 0.37 |
| FF5 + MOM | −1.63 % | −0.82 | 0.39 |

Against the market alone, BAB is a 5.7 % anomaly; against FF5 it is nothing. The work is
done by **RMW (+0.57) and CMA (+0.80)**: low-beta stocks are profitable firms that invest
conservatively. Nothing about the strategy changed, only the benchmark — the lecture's
+3.0 % CAPM alpha that became −0.4 % once the manager's style was priced.

## Export

Bundle the figures and the tables, in case you want them for your own notes.

In [ ]:
figures = {"sml": fig1, "bab_wealth": fig2}
tables = {"capm": capm, "factors": factors,
          "two_factor": two_factor, "spanning": spanning}

save_results(figures=figures, tables=tables, name="ex10")

## Your turn

Three questions, no new code. Each one changes a single line in a cell you have already
run, and asks you to say what will move before you look.

Work in your own copy (File, then Save a copy in Drive), and put every line back when you
are done, so that the commentary above still matches your output. Write your answers
down, two or three sentences each. We go through them in the exercise block on
Monday 23 November, after the discussion of exercise sheet 10.

### 1 · Predict

Lecture 11 asks how much data a t-statistic needs. Here is a first look. In the data cell,
set `START = "2000-01"`, which keeps only the last 26 years of the sample.

Before you run anything, write down:

- whether the fitted SML of Task 1 is still flatter than the theoretical one,
- whether BAB's CAPM alpha in Task 2 comes out larger or smaller than 5.67 %, and whether
  its t-statistic comes out larger or smaller than 2.34,
- whether FF5 still spans BAB in Task 3.

Then run **Runtime → Restart and run all** and check your three answers. Set `START` back
to `"1963-07"` when you are done.

### 2 · Change one input

In Task 2, BAB is built from the two extreme deciles. Build it from the next ones in
instead. In the first code cell of Task 2, replace

    LONG, SHORT = "Lo 10", "Hi 10"

with

    LONG, SHORT = "Dec 2", "Dec 9"

and run that cell and the cells after it. Using the CAPM alphas of the four deciles in
Task 1, explain in two sentences why BAB's alpha falls so much. Put the original line
back when you are done.

### 3 · Interpret

A pension fund trustee reads Task 2 and says:

*"BAB earns an alpha of 5.7 % a year with a t-statistic of 2.3. We put 10 % of our equity
allocation into it."*

Using Tasks 2 and 3, explain in three or four sentences what this strategy actually is,
and what holding it would mean for the pension fund.